# Clase 5 — La forma de los datos manda (I): imágenes

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** La arquitectura de una red **es una teoría sobre
> cómo está estructurado el mundo**. Una CNN afirma dos cosas: que lo que importa
> es *local*, y que un patrón *vale igual en cualquier posición*. Cuando eso es
> cierto, la red aprende mejor con menos perillas. Cuando es falso, te miente con
> mucha seguridad.

---

**El recorrido de hoy, en cuatro movimientos:**

1. **Repaso.** La red densa de la clase 4, con el mismo loop de cinco líneas,
   ahora mirando fotos de dígitos escritos a mano. Anda bien... hasta que corremos
   la foto unos píxeles.
2. **La convolución sin misterio.** Un pincel de nueve números que recorre la
   imagen. Primero con dos `for`, después con `nn.Conv2d`.
3. **La CNN.** Mismo loop, otra máquina: menos perillas, menos errores, y aguanta
   el corrimiento.
4. **Mirar adentro.** Qué pinceles aprendió, qué ilumina cada uno, y qué parte de
   la foto miró para decidir. Y con esas mismas herramientas, volvemos a un
   archivo fotográfico de ciencias sociales a ver qué está midiendo *de verdad*.

**Cuando termines este notebook vas a poder:**

1. Cargar imágenes en un `Dataset`, repartirlas con un `DataLoader` y escribir un
   modelo como una clase de PyTorch (`nn.Module`).
2. Mostrar, con un experimento propio, que una red densa **nunca supo** que sus
   entradas formaban una grilla.
3. Escribir una convolución a mano y verificar que `nn.Conv2d` da **el mismo
   número**.
4. Entrenar una CNN chica, compararla contra la densa, y mirar sus filtros, sus
   mapas de activación y sus mapas de "dónde miró".
5. Hacer análisis de errores sobre un clasificador de imágenes y decir **qué
   está midiendo realmente**.

**Lo que hace falta traer:** la clase 4 (el training loop, train/val/test).

**Cuánto tarda.** Todo el notebook corre en CPU en **unos 2 o 3 minutos**, y puede
irse a 5 en Colab. Lo más lento es entrenar la CNN (menos de un minuto). No hace
falta GPU.

**Cómo se usa.** Corré las celdas en orden (`Shift + Enter`). Donde diga `# TODO`
te toca a vos. Los chequeos de este notebook **nunca explotan**: te dicen en
castellano qué salió y qué probar. Si algo se rompe de verdad, mirá
`recursos/setup.md`.

## 0. Preparar el entorno

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/fixes-revision-clases"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla y un ayudante

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, random_split

from ssdl import graficos as g, utils

SEMILLA = 1983
utils.fijar_semilla(SEMILLA)       # la semilla del curso
utils.resumen_del_entorno()


def revisar(condicion, bien, pista):
    """El corrector amable: nunca levanta una excepción, siempre dice algo útil."""
    if condicion:
        print("OK ·", bien)
    else:
        print("Todavía no ·", pista)
    return bool(condicion)

---

## 1. El sandbox de hoy: MNIST

Hoy arrancamos en un **sandbox técnico**, y lo decimos con todas las letras: no es
un dato de ciencias sociales. Es MNIST, 70.000 dígitos escritos a mano, de 28×28
píxeles. Es el "hola mundo" de las imágenes: chico, rápido, y lo bastante difícil
como para que se note la diferencia entre una máquina y otra. En la sección 7
volvemos al dominio.

> **Un dato que les va a gustar.** Los dígitos de MNIST los escribieron
> **empleados de la Oficina del Censo** de Estados Unidos y **estudiantes
> secundarios**. Y la primera versión del dataset tenía un problema de
> partición: los empleados del censo en un lado y los estudiantes en el otro,
> con letras bastante distintas. Hubo que rearmarlo mezclando las dos fuentes.
> Es la escena de la clase 4 (*partí bien tus datos*) pasando en el dataset más
> famoso del mundo.

In [ ]:
import os
import pathlib
import sys

# datos/descargar.py sabe traer MNIST (y usar la cache si ya está)
RAIZ = pathlib.Path(os.environ["DLSS_RAIZ"])
sys.path.insert(0, str(RAIZ / "datos"))


def figuras_sinteticas(n, semilla=0):
    """Plan B sin red: cuatro clases de figuras de 28x28, hechas con numpy."""
    rng = np.random.default_rng(semilla)
    X = np.zeros((n, 28, 28), dtype=np.uint8)
    y = rng.integers(0, 4, size=n).astype(np.int64)
    for i in range(n):
        f, c = rng.integers(4, 12), rng.integers(4, 12)
        alto, ancho = rng.integers(9, 16), rng.integers(9, 16)
        img = np.zeros((28, 28))
        if y[i] == 0:                                    # barras verticales
            img[f:f + alto, c:c + ancho:3] = 1.0
        elif y[i] == 1:                                  # barras horizontales
            img[f:f + alto:3, c:c + ancho] = 1.0
        elif y[i] == 2:                                  # bloque lleno
            img[f:f + alto, c:c + ancho] = 1.0
        else:                                            # marco hueco
            img[f:f + alto, c:c + ancho] = 1.0
            img[f + 2:f + alto - 2, c + 2:c + ancho - 2] = 0.0
        X[i] = np.clip(img * 200 + rng.normal(0, 12, (28, 28)), 0, 255)
    return X, y


try:
    import descargar
    _tr, _te = descargar.mnist("train"), descargar.mnist("t10k")
    if _tr is None or _te is None:
        raise RuntimeError("sin datos")
    (X_train_todo, y_train_todo), (X_test_todo, y_test_todo) = _tr, _te
    NOMBRES = [str(d) for d in range(10)]
    FUENTE = "MNIST"
except Exception as e:                                   # noqa: BLE001
    print(f"[!] MNIST no disponible ({type(e).__name__}). Voy al plan B sintético.")
    X_train_todo, y_train_todo = figuras_sinteticas(12000, 1)
    X_test_todo, y_test_todo = figuras_sinteticas(4000, 2)
    NOMBRES = ["barras verticales", "barras horizontales", "bloque", "marco"]
    FUENTE = "figuras sintéticas"
    print("    Con el plan B el notebook corre entero, pero las figuras son mucho")
    print("    más fáciles que los dígitos: las dos redes van a acertar casi todo.")
    print("    El corrimiento y los mapas de 'dónde miró' sí se ven igual.")

print(f"\nFuente: {FUENTE}")
print(f"entrenamiento: {np.shape(X_train_todo)}  ->  {np.shape(X_train_todo)[0]} "
      f"imágenes de {np.shape(X_train_todo)[1]}x{np.shape(X_train_todo)[2]} en escala de grises")
print(f"test:          {np.shape(X_test_todo)}")

### Miralas

Antes de escribir una línea de modelo, mirá el dato. Es la primera regla de
cualquier análisis y con imágenes es literal.

In [ ]:
idx = np.arange(16)
fig, _ = g.mostrar_imagenes(
    X_train_todo[idx],
    etiquetas=[NOMBRES[int(y_train_todo[k])] for k in idx],
    ncols=8, titulo=f"16 imágenes de {FUENTE} (sandbox técnico)")
plt.show()

### Una imagen es una tabla de números

Esto es lo único que ve la computadora: una grilla de números del 0 (negro) al
255 (blanco). Recortamos un pedacito de 8×8 y lo imprimimos al lado, para que se
vea que el dibujo **es** la tabla.

In [ ]:
foto = X_train_todo[0]
recorte = foto[10:18, 8:16]        # ocho filas por ocho columnas del medio

fig, ejes = g.figura(11, 4.2, filas=1, columnas=2,
                     titulo="La misma cosa, dos veces: el dibujo y la tabla")
ejes[0].imshow(foto, cmap="gray")
ejes[0].add_patch(plt.Rectangle((7.5, 9.5), 8, 8, fill=False,
                                edgecolor=g.ACENTO, linewidth=2))
ejes[0].set_title(f"la foto entera: {foto.shape[0]}x{foto.shape[1]}")
ejes[1].imshow(recorte, cmap="gray", vmin=0, vmax=255)
for f in range(8):
    for c in range(8):
        v = int(recorte[f, c])
        ejes[1].text(c, f, str(v), ha="center", va="center", fontsize=7,
                     color="black" if v > 128 else "white")
ejes[1].set_title("el recorte de 8x8, con sus números")
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

### De la tabla a PyTorch: `Dataset` y `DataLoader`

En la clase 4 los datos eran 36 años: entraban todos juntos en cada paso. Acá
son 60.000 fotos, y PyTorch tiene dos piezas para manejar eso. Las vas a ver en
**todos** los tutoriales, así que conviene conocerlas por su nombre:

| pieza | qué es | la imagen |
|---|---|---|
| `Dataset` | la pila de fichas: `datos[i]` te da **una** foto y su etiqueta | el fichero del archivo |
| `DataLoader` | el que reparte: baraja la pila y te da las fichas **de a lotes** (*batches*) | quien te alcanza las cajas de a una |

El lote es la idea de la clase 4 (*epoch*, *batch*) hecha objeto: cada vuelta del
loop ve 64 fotos, no 60.000. Una época es haber pasado por todas las cajas.

Y la partición, como en la clase 4, se hace **antes** de mirar nada:

- **entrenamiento** (50.000): con estas se mueven las perillas;
- **validación** (10.000): con estas comparamos modelos y decidimos;
- **test** (los otros 10.000, que vienen aparte): **en un sobre cerrado**. Se
  abre una sola vez, en la sección 6.

> **Si buscás en internet** vas a ver que casi todos cargan MNIST así:
>
> ```python
> from torchvision import datasets, transforms
> datos = datasets.MNIST("datos", train=True, download=True,
>                        transform=transforms.ToTensor())
> ```
>
> Es lo mismo que hacemos acá a mano: un `Dataset` donde `datos[i]` devuelve una
> foto de `(1, 28, 28)` con valores entre 0 y 1, y su etiqueta. Lo armamos con
> `TensorDataset` para que funcione también sin `torchvision` y sin red.

In [ ]:
def a_tensor(X):
    """uint8 (n, 28, 28) -> float (n, 1, 28, 28) entre 0 y 1."""
    return torch.tensor(np.asarray(X) / 255.0, dtype=torch.float32).unsqueeze(1)


todo_train = TensorDataset(a_tensor(X_train_todo),
                           torch.tensor(np.asarray(y_train_todo), dtype=torch.long))
datos_test = TensorDataset(a_tensor(X_test_todo),
                           torch.tensor(np.asarray(y_test_todo), dtype=torch.long))

n_val = len(todo_train) // 6                        # 10.000 de 60.000
datos_train, datos_val = random_split(
    todo_train, [len(todo_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(SEMILLA))

cargador_train = DataLoader(datos_train, batch_size=64, shuffle=True,
                            generator=torch.Generator().manual_seed(SEMILLA))
cargador_val = DataLoader(datos_val, batch_size=1000)     # sin barajar: sólo medimos
cargador_test = DataLoader(datos_test, batch_size=1000)   # el sobre cerrado

N_CLASES = len(NOMBRES)
print(f"entrenamiento: {len(datos_train)}  ·  validación: {len(datos_val)}  ·  "
      f"test: {len(datos_test)} (cerrado)")

imagen, etiqueta = datos_train[0]
print("\nuna ficha del Dataset:  imagen", tuple(imagen.shape), "· etiqueta", int(etiqueta))

x_lote, y_lote = next(iter(cargador_train))
print("un lote del DataLoader: imágenes", tuple(x_lote.shape), "· etiquetas", tuple(y_lote.shape))
print("                        (batch, canales, alto, ancho)")

> **La cuaterna que vas a leer en todos los errores: `(batch, canales, alto,
> ancho)`.** Cuántas fotos, cuántas capas de color (1 en gris, 3 en color),
> cuán alta, cuán ancha. La mitad de los errores de esta clase se arreglan
> leyendo esa línea despacio.

---

## 2. Repaso: la red de la clase 4, ahora mirando fotos

La máquina es **la misma de la clase 4**: dos capas densas con un doblez (`ReLU`)
en el medio. Cambian tres cosas, y ninguna es de fondo:

1. **Entra una foto, no un número.** Adelante ponemos `nn.Flatten`, que estira la
   grilla de 28×28 en una fila de 784 números.
2. **Salen 10 números, no uno.** Uno por dígito: el puntaje de "esto es un 0",
   "esto es un 1", etc. El más alto gana.
3. **La loss es otra.** Para elegir entre 10 opciones no usamos el error
   cuadrático de la clase 3 sino `CrossEntropyLoss`: convierte los 10 puntajes en
   probabilidades y mide **cuánto se sorprendió** el modelo con la respuesta
   correcta. Si le daba 90 % al dígito correcto, poca sorpresa, loss chica. Si le
   daba 1 %, mucha.

### Escribir el modelo como una clase

Hasta ahora usamos `nn.Sequential`. Los tutoriales (y la CNN de más abajo)
escriben los modelos así, como una **clase** con dos partes:

- `__init__`: **las perillas**. Qué capas tiene la máquina.
- `forward`: **el dibujo**. Por dónde pasa el dato, en qué orden.

Es la misma red; sólo cambia cómo se escribe. Y tiene una ventaja que vamos a
usar en la sección 5: cada capa tiene **nombre** (`self.capa1`), así que
después podemos ir a mirarla.

In [ ]:
class RedDensa(nn.Module):
    """La red de la clase 4 (dos capas densas y un doblez), comiendo píxeles."""

    def __init__(self, n_clases=10):
        super().__init__()
        self.aplanar = nn.Flatten()               # 28x28  ->  784 números en fila
        self.capa1 = nn.Linear(28 * 28, 64)
        self.capa2 = nn.Linear(64, n_clases)

    def forward(self, x):
        x = self.aplanar(x)
        x = torch.relu(self.capa1(x))             # el doblez de la clase 2
        return self.capa2(x)                      # 10 puntajes, uno por clase


torch.manual_seed(SEMILLA)
densa = RedDensa(N_CLASES)
print(densa)
print("\nperillas:", f"{utils.contar_parametros(densa):,}".replace(",", "."))

### Antes de entrenar: ¿qué opina?

Le mostramos una foto a la red recién nacida. `torch.softmax` convierte los 10
puntajes en probabilidades que suman 1.

In [ ]:
X_val = torch.stack([datos_val[i][0] for i in range(len(datos_val))])
y_val = torch.stack([datos_val[i][1] for i in range(len(datos_val))])


def barras_de_probabilidad(ax, modelo, x, titulo=""):
    """Las 10 probabilidades que le da el modelo a UNA foto x de (1, 28, 28)."""
    modelo.eval()
    with torch.no_grad():
        p = torch.softmax(modelo(x.unsqueeze(0)), dim=1)[0].numpy()
    colores = [g.MODELO] * len(p)
    colores[int(p.argmax())] = g.ACENTO
    ax.bar(range(len(p)), p * 100, color=colores)
    ax.set_xticks(range(len(p)))
    ax.set_xticklabels(NOMBRES if len(NOMBRES) == 10 else range(len(p)))
    ax.set_ylim(0, 100)
    ax.set_ylabel("%")
    ax.set_title(titulo)


def foto_y_barras(modelo, x, titulo):
    fig, ejes = g.figura(10, 3.2, filas=1, columnas=2, titulo=titulo,
                         gridspec_kw={"width_ratios": [1, 3]})
    ejes[0].imshow(x[0], cmap="gray")
    ejes[0].set_xticks([]); ejes[0].set_yticks([]); ejes[0].grid(False)
    barras_de_probabilidad(ejes[1], modelo, x, "probabilidad de cada dígito")
    plt.show()


ejemplo = X_val[0]
print("la respuesta correcta es:", NOMBRES[int(y_val[0])])
foto_y_barras(densa, ejemplo, "La red recién nacida: no tiene idea")

Todas parecidas, cerca del 10 %: está adivinando. Las perillas son números al
azar. Ahora el loop.

### El loop de la clase 4, con un `DataLoader`

Son **las mismas cinco líneas**: predecir, comparar, borrar, anotar, mover. Lo
único nuevo es el `for` de afuera, que le pide los lotes al `DataLoader`.

Y una función aparte para medir sobre validación, con las dos cosas de la clase 4
que no hay que olvidar: `modelo.eval()` y `torch.no_grad()` (acá no movemos
perillas, sólo miramos).

In [ ]:
perdida = nn.CrossEntropyLoss()


def entrenar_una_epoca(modelo, cargador, optimizador):
    modelo.train()
    suma = 0.0
    for x, y in cargador:                      # el DataLoader reparte los lotes
        prediccion = modelo(x)                 # 1. PREDECIR
        l = perdida(prediccion, y)             # 2. COMPARAR
        optimizador.zero_grad()                # 3. BORRAR
        l.backward()                           # 4. ANOTAR
        optimizador.step()                     # 5. MOVER
        suma += l.item() * len(y)
    return suma / len(cargador.dataset)


def evaluar(modelo, cargador):
    """Devuelve (loss, % de aciertos) sobre datos que el modelo no usa para aprender."""
    modelo.eval()
    suma, aciertos = 0.0, 0
    with torch.no_grad():
        for x, y in cargador:
            prediccion = modelo(x)
            suma += perdida(prediccion, y).item() * len(y)
            aciertos += (prediccion.argmax(dim=1) == y).sum().item()
    n = len(cargador.dataset)
    return suma / n, 100 * aciertos / n


def entrenar(modelo, epocas=4, lr=1e-3):
    """Entrena y va anotando las dos curvas de la clase 4."""
    torch.manual_seed(SEMILLA)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    historial = {"train": [], "val": [], "exactitud": []}
    for epoca in range(1, epocas + 1):
        loss_train = entrenar_una_epoca(modelo, cargador_train, optimizador)
        loss_val, exactitud = evaluar(modelo, cargador_val)
        historial["train"].append(loss_train)
        historial["val"].append(loss_val)
        historial["exactitud"].append(exactitud)
        print(f"época {epoca}:  loss train {loss_train:.3f} · loss val {loss_val:.3f} "
              f"· aciertos en validación {exactitud:.1f} %")
    return historial

In [ ]:
import time

t0 = time.time()
hist_densa = entrenar(densa)
print(f"\n({time.time() - t0:.0f} segundos)")

g.curvas_de_entrenamiento(train=hist_densa["train"], val=hist_densa["val"],
                          titulo="La red densa: las curvas de la clase 4")
plt.show()

In [ ]:
foto_y_barras(densa, ejemplo, "La misma foto, después de entrenar")

### Los errores: mirarlos, no contarlos

Un porcentaje no te dice **en qué** se equivoca. Mirá 16 errores. En rojo, lo que
dijo; entre paréntesis, lo que era.

In [ ]:
def predecir(modelo, X):
    modelo.eval()
    with torch.no_grad():
        return modelo(X).argmax(dim=1)


pred_densa = predecir(densa, X_val)
errores_densa = torch.where(pred_densa != y_val)[0]
print(f"la densa se equivocó en {len(errores_densa)} de {len(y_val)} fotos de validación")

k = errores_densa[:16]
g.mostrar_imagenes(X_val[k], etiquetas=[NOMBRES[int(i)] for i in y_val[k]],
                   predicciones=[NOMBRES[int(i)] for i in pred_densa[k]],
                   titulo="16 errores de la red densa")
plt.show()

### ¿Qué aprendió la densa? Plantillas pegadas a un lugar

Cada una de las 64 neuronas de la primera capa tiene **784 perillas, una por
píxel**. Si las reacomodamos en una grilla de 28×28 podemos *ver* qué busca cada
neurona: rojo, "acá quiero tinta"; azul, "acá no".

In [ ]:
pesos = densa.capa1.weight.detach().reshape(-1, 28, 28)     # (64, 28, 28)
g.grilla_de_filtros(pesos[:16], ncols=8,
                    titulo="Lo que busca cada neurona de la densa: una plantilla del tamaño de la foto")
plt.show()

> **Fijate en una cosa.** Cada neurona mira la foto **entera**, y cada perilla
> está atada a **un píxel en particular**. Lo que aprendió son plantillas
> borrosas *pegadas a un lugar de la hoja*: "tinta en esta zona, nada en esta
> otra". Se parece más a **calcar** sobre un molde fijo que a reconocer un trazo.
>
> Si eso es así, hay una pregunta incómoda.

### `TODO 1` — apostá primero

Vamos a tomar las 10.000 fotos de validación y **correrlas 4 píxeles a la
derecha**. Para vos son los mismos dígitos. Nadie reentrena nada.

**Antes de correr la celda de abajo, escribí tu apuesta:** ¿qué porcentaje va a
acertar la red densa con las fotos corridas? Hoy acierta alrededor del 96 %.

Sin la apuesta previa, el resultado es sólo un número más.

In [ ]:
# TODO 1: escribí tu apuesta (un número entre 0 y 100) ANTES de correr la celda siguiente.
mi_apuesta = None

_ = revisar(mi_apuesta is not None and 0 <= mi_apuesta <= 100,
        f"apostaste {mi_apuesta} %. Ahora sí, corré la celda que sigue",
        "escribí un número en mi_apuesta: cualquier número, pero ANTES de mirar")

In [ ]:
def correr(X, d):
    """La misma imagen, corrida d píxeles a la derecha. Por la izquierda entra
    fondo, no el otro borde: una foto corrida no da la vuelta."""
    if d == 0:
        return X
    Y = torch.zeros_like(X)
    Y[..., d:] = X[..., :-d]
    return Y


def exactitud_corrida(modelo, d):
    return 100 * (predecir(modelo, correr(X_val, d)) == y_val).float().mean().item()


DESPLAZAMIENTOS = [0, 1, 2, 3, 4, 5, 6]
corr_densa = [exactitud_corrida(densa, d) for d in DESPLAZAMIENTOS]
for d, v in zip(DESPLAZAMIENTOS, corr_densa):
    print(f"corrida {d} px:  la densa acierta {v:5.1f} %")
if mi_apuesta is not None:
    print(f"\ntu apuesta para 4 px: {mi_apuesta} %  ·  lo que pasó: {corr_densa[4]:.1f} %")

foto_y_barras(densa, correr(ejemplo, 4), "La misma foto, corrida 4 píxeles: la densa se confunde")

> **Lo que acaba de pasar.** Para vos, correr la foto no cambió nada. Para la
> densa cambió **todo**: cada píxel es una variable distinta, y le movimos todas
> las variables al mismo tiempo. Sus plantillas estaban pegadas a un lugar de la
> hoja, y el dígito ya no está ahí.
>
> La red densa **nunca supo que había una grilla**. Para ella, el píxel de la
> fila 12, columna 10 no es "vecino" del de la columna 11: es la variable 346 y
> la variable 347, tan vecinas como la 3 y la 700. El problema de hoy es cómo
> meterle a una máquina algo que para nosotros es obvio: que **un trazo es el
> mismo trazo un poco más a la derecha**.

---

## 3. La convolución sin misterio

Una convolución es esto y nada más: una ventanita de 3×3 que recorre la imagen;
en cada posición multiplica lo que ve por nueve números y suma. Ese resultado es
un píxel del **mapa de activación**, que es un dibujo nuevo.

**Vocabulario firme, porque se confunde todos los años:**

| palabra | qué es |
|---|---|
| **filtro** (o *kernel*) | los nueve números. El pincel. |
| **mapa de activación** | el dibujo que sale. El cuadro pintado con ese pincel. |

Escribimos la convolución con dos `for`, a propósito lenta y transparente. Es la
misma cuenta que hace la viz `kernel-deslizante.html` y la misma que hace
PyTorch.

In [ ]:
def convolucion(imagen, filtro):
    """La convolución con dos for anidados. Lenta a propósito: se ve la cuenta.

    imagen: (alto, ancho) en numpy · filtro: (3, 3)
    devuelve: (alto-2, ancho-2)
    """
    alto, ancho = imagen.shape
    kf, kc = filtro.shape
    salida = np.zeros((alto - kf + 1, ancho - kc + 1))
    for f in range(salida.shape[0]):
        for c in range(salida.shape[1]):
            parche = imagen[f:f + kf, c:c + kc]     # lo que ve la ventana
            salida[f, c] = np.sum(parche * filtro)  # multiplicar y sumar
    return salida


foto_gris = X_train_todo[0] / 255.0

BORDE_VERTICAL = np.array([[1, 0, -1],
                           [2, 0, -2],
                           [1, 0, -1]], dtype=float)
BORDE_HORIZONTAL = BORDE_VERTICAL.T
DESENFOQUE = np.ones((3, 3)) / 9.0

mapas = {"borde vertical": convolucion(foto_gris, BORDE_VERTICAL),
         "borde horizontal": convolucion(foto_gris, BORDE_HORIZONTAL),
         "desenfoque": convolucion(foto_gris, DESENFOQUE)}

fig, ejes = g.figura(13, 3.6, filas=1, columnas=4,
                     titulo="La misma foto, pintada con tres pinceles distintos")
ejes[0].imshow(foto_gris, cmap="gray"); ejes[0].set_title("la foto")
for ax, (nombre, mapa) in zip(ejes[1:], mapas.items()):
    lim = np.abs(mapa).max()
    ax.imshow(mapa, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(nombre)
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

print("forma de la foto: ", foto_gris.shape)
print("forma del mapa:   ", mapas["borde vertical"].shape,
      " <- pierde un píxel de cada lado: la ventana no puede asomarse afuera")
print("(la CNN de la sección 4 le pone un marco de ceros, padding=1, y no se achica)")

### La ventana, recorriendo

La misma cuenta, en cámara lenta: arriba, dónde está la ventana; abajo, cuánto
del mapa de activación lleva pintado hasta ese momento.

In [ ]:
mapa_completo = convolucion(foto_gris, BORDE_VERTICAL)
alto_m, ancho_m = mapa_completo.shape
momentos = [(5, 8), (12, 3), (18, 14), (alto_m - 1, ancho_m - 1)]   # (fila, columna)
lim = np.abs(mapa_completo).max()

fig, ejes = g.figura(13, 6.4, filas=2, columnas=4,
                     titulo="El pincel de bordes verticales recorriendo la foto, fila por fila")
for j, (f, c) in enumerate(momentos):
    arriba, abajo = ejes[0, j], ejes[1, j]
    arriba.imshow(foto_gris, cmap="gray")
    arriba.add_patch(plt.Rectangle((c - 0.5, f - 0.5), 3, 3, fill=False,
                                   edgecolor=g.ACENTO, linewidth=2.5))
    arriba.set_title(f"ventana en ({f}, {c})")
    parcial = np.full_like(mapa_completo, np.nan)
    hasta = f * ancho_m + c + 1                       # cuántas posiciones ya recorrió
    parcial.flat[:hasta] = mapa_completo.flat[:hasta]
    abajo.imshow(parcial, cmap="RdBu_r", vmin=-lim, vmax=lim)
    abajo.add_patch(plt.Rectangle((c - 0.5, f - 0.5), 1, 1, fill=False,
                                  edgecolor=g.ACENTO, linewidth=2))
    abajo.set_title(f"mapa: {hasta} de {mapa_completo.size} píxeles")
for ax in ejes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

f, c = momentos[0]
parche = foto_gris[f:f + 3, c:c + 3]
print("en la primera posición, la ventana ve:\n", parche.round(2))
print("multiplicado por el filtro y sumado:", round(float(np.sum(parche * BORDE_VERTICAL)), 3),
      "  <- ese es UN píxel del mapa")

### El mismo pincel en todos lados

Acá está la clave de toda la clase. El filtro es **el mismo** en cada posición:
nueve números, no nueve números *por lugar*. Entonces, si corrés la foto, el
mapa de activación... se corre igual. El trazo que encontraba acá, lo encuentra
allá.

In [ ]:
foto_corrida = correr(torch.tensor(foto_gris), 4).numpy()
mapa_original = convolucion(foto_gris, BORDE_VERTICAL)
mapa_de_la_corrida = convolucion(foto_corrida, BORDE_VERTICAL)

fig, ejes = g.figura(12, 6.2, filas=2, columnas=2,
                     titulo="Corrés la foto, se corre el mapa: el pincel no sabe dónde está")
ejes[0, 0].imshow(foto_gris, cmap="gray"); ejes[0, 0].set_title("la foto")
ejes[0, 1].imshow(foto_corrida, cmap="gray"); ejes[0, 1].set_title("corrida 4 píxeles")
ejes[1, 0].imshow(mapa_original, cmap="RdBu_r", vmin=-lim, vmax=lim)
ejes[1, 0].set_title("su mapa de bordes verticales")
ejes[1, 1].imshow(mapa_de_la_corrida, cmap="RdBu_r", vmin=-lim, vmax=lim)
ejes[1, 1].set_title("el mapa de la corrida")
for ax in ejes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

diferencia = np.abs(mapa_de_la_corrida[:, 4:] - mapa_original[:, :-4]).max()
print(f"¿el mapa de la foto corrida es el mapa original, corrido? "
      f"diferencia máxima: {diferencia:.1e}")

> **Esto es lo que le faltaba a la densa.** La densa tenía una perilla *por
> píxel*: lo que aprendía en un lugar no le servía en otro. La convolución tiene
> un pincel *para toda la foto*: lo que aprende en una esquina vale en todas las
> esquinas. Eso es un **supuesto sobre el mundo**, metido en la forma de la
> máquina.

### `TODO 2` — inventá un detector de diagonales

Mirá el filtro de bordes verticales: tiene los positivos de un lado y los
negativos del otro, y por eso se enciende donde hay un cambio **de izquierda a
derecha**.

*Pista 1: un detector de diagonales tiene que tener la zona positiva de un lado
de la diagonal y la negativa del otro.*
*Pista 2: probá poner los positivos en la esquina de arriba a la izquierda y los
negativos en la de abajo a la derecha.*
*Pista 3: la suma de los nueve números conviene que dé cero, si no el filtro
también cambia el brillo general.*

> **Ojo con una trampa que confunde a todo el mundo.** Un filtro con **+ de un
> lado y − del otro** detecta un **borde**, y el borde que detecta es
> **perpendicular** a la dirección en que cambia el signo. El de bordes
> verticales cambia de izquierda a derecha y se enciende en líneas verticales.
> Con la pista 2 (+ arriba a la izquierda, − abajo a la derecha) el signo cambia
> en la dirección «↘», así que se enciende en los bordes de la **otra**
> diagonal, la «/». Sobre una raya «\\» da cero, y está bien que dé cero.
>
> Si lo que querés es encender una **raya** «\\» (no un borde), el patrón es
> otro: positivos *sobre* la diagonal y negativos a los costados, por ejemplo
> `[[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]`. El chequeo de abajo acepta
> cualquiera de los dos y te dice a cuál de las dos rayas reacciona más el tuyo.

In [ ]:
# TODO 2: cambiá estos nueve números para que detecte diagonales.
mi_filtro = np.array([[0, 0, 0],
                      [0, 0, 0],
                      [0, 0, 0]], dtype=float)

mapa_mio = convolucion(foto_gris, mi_filtro)

fig, ejes = g.figura(9, 3.8, filas=1, columnas=3, titulo="Tu filtro en acción")
ejes[0].imshow(foto_gris, cmap="gray"); ejes[0].set_title("la foto")
lim = np.abs(mi_filtro).max() or 1
ejes[1].imshow(mi_filtro, cmap="RdBu_r", vmin=-lim, vmax=lim)
ejes[1].set_title("tus nueve números")
for f in range(3):
    for c in range(3):
        ejes[1].text(c, f, f"{mi_filtro[f, c]:g}", ha="center", va="center", fontsize=11)
lim2 = np.abs(mapa_mio).max() or 1
ejes[2].imshow(mapa_mio, cmap="RdBu_r", vmin=-lim2, vmax=lim2)
ejes[2].set_title("el mapa de activación")
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

# --- el chequeo: se lo probamos contra dos imágenes hechas a propósito -------
diagonal = np.eye(9)                        # una raya de arriba-izq a abajo-der
antidiagonal = np.fliplr(np.eye(9))         # la raya al revés
r_diag = np.abs(convolucion(diagonal, mi_filtro)).sum()
r_anti = np.abs(convolucion(antidiagonal, mi_filtro)).sum()

_ = revisar(np.abs(mi_filtro).sum() > 0,
        "el filtro ya no es todo ceros",
        "sigue en todo ceros: todo por cero es cero y el mapa sale vacío")
_ = revisar(abs(mi_filtro.sum()) < 1e-9,
        "los nueve números suman cero: no te cambia el brillo general",
        f"suman {mi_filtro.sum():g}. No está mal, pero además de detectar bordes "
        "te aclara o te oscurece la imagen entera")
if r_diag != r_anti:
    raya = "«\\» (la de np.eye)" if r_diag > r_anti else "«/»"
    print(f"tu filtro reacciona más a la raya {raya} ({r_diag:.1f} contra {r_anti:.1f})")
_ = revisar(r_diag > 1.35 * r_anti or r_anti > 1.35 * r_diag,
        "distingue una diagonal de la otra: eso es un detector de diagonales",
        f"reacciona casi igual a las dos diagonales ({r_diag:.1f} vs {r_anti:.1f}). "
        "Probá con los positivos arriba a la izquierda y los negativos abajo a la derecha")

### Opcional — el filtro que no hace nada

Un ejercicio de un minuto que consolida la mecánica.

*Pista: si en cada posición querés que el resultado sea el píxel del centro y
nada más, ¿cuánto tiene que valer cada uno de los nueve números? Pensá en la
multiplicación por 1 y en la multiplicación por 0.*

In [ ]:
# Opcional: escribí el filtro que deja la imagen igualita (la "identidad").
filtro_identidad = np.array([[0, 0, 0],
                             [0, 0, 0],
                             [0, 0, 0]], dtype=float)

salida = convolucion(foto_gris, filtro_identidad)
centro = foto_gris[1:-1, 1:-1]

_ = revisar(np.allclose(salida, centro),
        "sale la imagen igual (menos el borde de un píxel, que la ventana no alcanza)",
        "todavía no devuelve la imagen. Acordate: querés quedarte SÓLO con el "
        "píxel del centro de la ventana, así que ocho de los nueve números tienen "
        "que anular lo que ven")

---

### `nn.Conv2d`: lo mismo, pero rápido

Regla de la casa: **ninguna abstracción antes de haber hecho a mano lo que
esconde**. Ya escribiste la convolución con dos `for`; ahora la reemplazamos por
la de PyTorch, y **verificamos que dan el mismo número**. Si no diera lo mismo,
sería que una de las dos está mal.

In [ ]:
# la misma cuenta, ahora en torch. OJO con las dimensiones:
#   la imagen entra como (batch, canales, alto, ancho)
#   el filtro entra como (filtros_de_salida, canales_de_entrada, alto, ancho)
capa = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    capa.weight[:] = torch.tensor(BORDE_VERTICAL, dtype=torch.float32).reshape(1, 1, 3, 3)

entrada = torch.tensor(foto_gris, dtype=torch.float32).reshape(1, 1, 28, 28)
with torch.no_grad():
    salida_torch = capa(entrada)[0, 0].numpy()

salida_a_mano = convolucion(foto_gris, BORDE_VERTICAL)

print("a mano  ->", salida_a_mano.shape, " torch ->", salida_torch.shape)
print("diferencia máxima entre las dos:", float(np.abs(salida_a_mano - salida_torch).max()))
_ = revisar(np.allclose(salida_a_mano, salida_torch, atol=1e-5),
        "el doble for y nn.Conv2d dan lo mismo (la diferencia es de redondeo: "
        "menos de una millonésima). No hay magia adentro",
        "dan distinto: revisá la forma del filtro (1, 1, 3, 3) y que sea el mismo filtro")

> **Leer un error de forma es contenido, no un accidente.** Si te olvidás el
> `unsqueeze` o le pasás una imagen de 28×28 pelada, PyTorch te dice algo como
> *"expected 4-dimensional input... but got 2-dimensional input"*. Ese mensaje
> te está diciendo la cuaterna que falta: **(batch, canales, alto, ancho)** —
> cuántas fotos, cuántas capas de color, cuán alta, cuán ancha. La mitad de los
> errores de esta clase se arreglan leyendo esa línea despacio.

---

## 4. La CNN: mismo loop, otra máquina

Una CNN chiquita apila tres veces la misma receta, y recién al final decide:

| pieza | qué hace | la imagen |
|---|---|---|
| `nn.Conv2d` | 16, 32, 64 pinceles de 3×3 que recorren la foto | los pinceles |
| `ReLU` | el doblez de la clase 2, sobre cada mapa | |
| `nn.MaxPool2d(2)` | achica el mapa a la mitad, quedándose con el valor más fuerte de cada cuadradito de 2×2 | mirar desde más lejos |
| `nn.AdaptiveMaxPool2d(1)` | de cada mapa se queda con **un solo número**: el máximo | **"¿apareció esta pieza en ALGÚN lado?"** |
| `nn.Linear` | con esos 64 números, decide | la densa que decide, al final |

Fijate en la penúltima: al final no importa **dónde** apareció cada pieza, sólo
**si** apareció. Ese es el supuesto de la clase convertido en una línea de código.

La escribimos como clase, igual que la densa. El método `por_dentro` devuelve los
mapas de activación de cada capa: lo vamos a usar en la sección 5 para mirar
adentro.

In [ ]:
class RedConvolucional(nn.Module):
    """Mira de a 3x3 tres veces, cada vez desde más lejos; al final se queda con
    el máximo de cada mapa ('¿apareció esta pieza en ALGÚN lado?') y decide."""

    def __init__(self, n_clases=10, final="maximo"):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 16 pinceles
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 32 pinceles
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 64 pinceles
        self.achicar = nn.MaxPool2d(2)
        if final == "maximo":
            self.resumir = nn.AdaptiveMaxPool2d(1)   # (64, 7, 7) -> (64, 1, 1)
            entradas = 64
        else:                                        # "aplanar": lo de la densa
            self.resumir = nn.Identity()             # (64, 7, 7) se queda igual
            entradas = 64 * 7 * 7
        self.decidir = nn.Linear(entradas, n_clases)

    def por_dentro(self, x):
        """Los mapas de activación de las tres capas, para poder mirarlos."""
        a1 = torch.relu(self.conv1(x))                     # (16, 28, 28)
        a2 = torch.relu(self.conv2(self.achicar(a1)))      # (32, 14, 14)
        a3 = torch.relu(self.conv3(self.achicar(a2)))      # (64,  7,  7)
        return a1, a2, a3

    def forward(self, x):
        a1, a2, a3 = self.por_dentro(x)
        resumen = torch.flatten(self.resumir(a3), start_dim=1)
        return self.decidir(resumen)


torch.manual_seed(SEMILLA)
cnn = RedConvolucional(N_CLASES)

perillas_densa = utils.contar_parametros(densa)
perillas_cnn = utils.contar_parametros(cnn)
print("perillas de la densa:", f"{perillas_densa:,}".replace(",", "."))
print("perillas de la CNN:  ", f"{perillas_cnn:,}".replace(",", "."),
      f"  <- menos de la mitad ({perillas_cnn / perillas_densa:.0%})")

### Seguí la forma

Un ejercicio de lectura que ahorra horas de errores: le pasamos un lote de 8
fotos y miramos la cuaterna `(batch, canales, alto, ancho)` después de cada paso.

In [ ]:
x = X_val[:8]
print(f"{'entrada':28s} {tuple(x.shape)}")
x = torch.relu(cnn.conv1(x));  print(f"{'conv1 + ReLU (16 pinceles)':28s} {tuple(x.shape)}")
x = cnn.achicar(x);            print(f"{'achicar a la mitad':28s} {tuple(x.shape)}")
x = torch.relu(cnn.conv2(x));  print(f"{'conv2 + ReLU (32 pinceles)':28s} {tuple(x.shape)}")
x = cnn.achicar(x);            print(f"{'achicar a la mitad':28s} {tuple(x.shape)}")
x = torch.relu(cnn.conv3(x));  print(f"{'conv3 + ReLU (64 pinceles)':28s} {tuple(x.shape)}")
x = cnn.resumir(x);            print(f"{'máximo de cada mapa':28s} {tuple(x.shape)}")
x = torch.flatten(x, 1);       print(f"{'aplanar':28s} {tuple(x.shape)}")
x = cnn.decidir(x);            print(f"{'decidir':28s} {tuple(x.shape)}  <- 10 puntajes por foto")

### Entrenarla: **la misma función** de la sección 2

No cambiamos ni una línea del loop. Lo único distinto es la máquina que le
pasamos. (Esto tarda menos de un minuto: la convolución hace muchas más cuentas
que la densa, aunque tenga menos perillas.)

In [ ]:
t0 = time.time()
hist_cnn = entrenar(cnn)
print(f"\n({time.time() - t0:.0f} segundos)")

fig, ax = g.figura(8.5, 4.2, titulo="Aciertos en validación, época por época",
                   xlabel="época", ylabel="% de aciertos")
epocas = range(1, len(hist_cnn["exactitud"]) + 1)
ax.plot(epocas, hist_densa["exactitud"], "o-", color=g.DATOS, label="red densa (clase 4)")
ax.plot(epocas, hist_cnn["exactitud"], "o-", color=g.MODELO, label="CNN")
ax.legend()
plt.show()

In [ ]:
pred_cnn = predecir(cnn, X_val)
errores_cnn = torch.where(pred_cnn != y_val)[0]
n = len(y_val)

print(f"{'':8s} {'perillas':>9s} {'aciertos':>9s} {'errores cada 1000 fotos':>25s}")
for nombre, perillas, pred in [("densa", perillas_densa, pred_densa),
                               ("CNN", perillas_cnn, pred_cnn)]:
    err = (pred != y_val).sum().item()
    print(f"{nombre:8s} {perillas:9,d} {100 * (1 - err / n):8.1f}% {1000 * err / n:25.0f}"
          .replace(",", "."))

_ = revisar(len(errores_cnn) < len(errores_densa),
        f"la CNN se equivoca en {len(errores_cnn)} fotos y la densa en "
        f"{len(errores_densa)}: con menos de la mitad de las perillas, "
        f"{len(errores_densa) / max(len(errores_cnn), 1):.1f} veces menos errores",
        "en esta corrida la CNN no mejoró a la densa. Pasa con el plan B sintético "
        "(es demasiado fácil); con MNIST no debería")

### Y el corrimiento, otra vez

La prueba que la densa no pasó. Los mismos dos modelos, las mismas fotos
corridas.

In [ ]:
corr_cnn = [exactitud_corrida(cnn, d) for d in DESPLAZAMIENTOS]

fig, ax = g.figura(8.5, 4.2, titulo="Qué pasa cuando la foto se corre unos píxeles",
                   xlabel="píxeles de corrimiento", ylabel="% de aciertos")
ax.plot(DESPLAZAMIENTOS, corr_densa, "o-", color=g.DATOS, label="red densa (clase 4)")
ax.plot(DESPLAZAMIENTOS, corr_cnn, "o-", color=g.MODELO, label="CNN")
ax.set_ylim(0, 100)
ax.legend()
plt.show()

for d, a, b in zip(DESPLAZAMIENTOS, corr_densa, corr_cnn):
    print(f"corrida {d} px:  densa {a:5.1f} %   ·   CNN {b:5.1f} %")

> **Esto es lo que compramos con el supuesto.** La densa se cae por el
> precipicio; la CNN casi ni se entera. Y ojo con la contracara: si la seguís
> corriendo, la CNN también termina cayendo (el dígito se empieza a salir de la
> foto). La invariancia no es mágica ni infinita: es un supuesto que se cumple
> *hasta cierto punto*.

### `TODO 3` — ¿de dónde sale la invariancia?

Hay dos sospechosos: los **pinceles** (que son los mismos en toda la foto) y el
**máximo del final** (que tira a la basura el *dónde*). ¿Cuál de los dos es el
que salva a la CNN del corrimiento?

Hacé el experimento: entrená la misma CNN pero con `final="aplanar"`. Así, en vez
de quedarse con el máximo de cada mapa, estira los 64 mapas de 7×7 en una fila
de 3.136 números y se los pasa a una capa densa, como hacía la red de la clase 4.

**Antes de correr, apostá:** ¿va a aguantar el corrimiento como la CNN, o se va a
caer como la densa? (Tarda unos 30 segundos.)

In [ ]:
# TODO 3: cambiá "maximo" por "aplanar" y corré la celda.
mi_variante = "maximo"

torch.manual_seed(SEMILLA)
cnn_variante = RedConvolucional(N_CLASES, final=mi_variante)
hist_variante = entrenar(cnn_variante)
corr_variante = [exactitud_corrida(cnn_variante, d) for d in DESPLAZAMIENTOS]

fig, ax = g.figura(8.5, 4.2, titulo="¿Quién aguanta el corrimiento?",
                   xlabel="píxeles de corrimiento", ylabel="% de aciertos")
ax.plot(DESPLAZAMIENTOS, corr_densa, "o-", color=g.DATOS, label="red densa")
ax.plot(DESPLAZAMIENTOS, corr_cnn, "o-", color=g.MODELO, label="CNN con máximo al final")
ax.plot(DESPLAZAMIENTOS, corr_variante, "s--", color=g.PARAMETRO,
        label=f"CNN con final='{mi_variante}'")
ax.set_ylim(0, 100)
ax.legend()
plt.show()

_ = revisar(mi_variante == "aplanar",
        f"con 4 px de corrimiento: CNN con máximo {corr_cnn[4]:.0f} %, "
        f"CNN aplanada {corr_variante[4]:.0f} %, densa {corr_densa[4]:.0f} %",
        "todavía estás entrenando la misma CNN de antes: cambiá mi_variante a \"aplanar\"")

> **Qué muestra esto.** Los pinceles solos no alcanzan: los mapas de
> activación se corren con la foto (lo viste en la sección 3), pero si al final
> los aplanás y se los das a una densa, esa densa vuelve a tener una perilla
> *por lugar*, y el problema vuelve. La aplanada aguanta bastante más que la
> densa, porque los `MaxPool2d` del medio ya perdonan corrimientos chiquitos;
> pero se cae igual. Recién el máximo del final dice, en serio, **"no me
> importa dónde"**.
>
> Y mirá las épocas: **sin** corrimiento, la aplanada acierta lo mismo o un
> poquito más que la CNN con máximo. El supuesto no es gratis: cuando el lugar
> sí importa (y en dígitos centrados, un poco importa), tirarlo a la basura
> cuesta. Lo que compra es robustez ante un cambio que **vos** sabés que va a
> pasar.
>
> Moraleja metodológica: **cada pieza de una arquitectura es un supuesto**, y se
> puede poner a prueba de a uno, como cualquier otro supuesto de un modelo.

---

## 5. Mirar adentro: qué ve la CNN

La densa tenía plantillas del tamaño de la foto. ¿Y la CNN? Cuatro maneras de
mirar, de la más directa a la más indirecta.

### 5.1 Los pinceles que aprendió sola

*(Esta es la celda que promete la visualización `viz/jerarquia.html`: allá ves un
esquema ilustrativo; acá ves los filtros de verdad de nuestra propia CNN.)*

Los nueve números de cada filtro son **parámetros**: se aprenden con la misma
loss y el mismo descenso por gradiente de la clase 3. Nadie los escribió.

In [ ]:
g.grilla_de_filtros(cnn.conv1.weight.detach(), ncols=8,
                    titulo="Los 16 pinceles de la primera capa, aprendidos por descenso por gradiente")
plt.show()

### 5.2 Qué ilumina cada pincel

Cada pincel pinta su propio cuadro de la misma foto. Abajo, los 16 mapas de la
primera capa: dónde se "encendió" cada pincel. Y después, 16 de los 32 de la
segunda capa, que ya no miran píxeles sino **los mapas de la primera**: combinan
bordes en pedazos de trazo.

In [ ]:
x = X_val[:1]
with torch.no_grad():
    a1, a2, a3 = cnn.por_dentro(x)

g.mostrar_imagenes(x[0].numpy(), etiquetas=["la foto"], ncols=1)
plt.show()
g.mostrar_imagenes(a1[0].numpy(), ncols=8, cmap="magma",
                   etiquetas=[f"pincel #{k}" for k in range(16)],
                   titulo="Capa 1 (28x28): 16 dibujos distintos de la misma foto")
plt.show()
g.mostrar_imagenes(a2[0, :16].numpy(), ncols=8, cmap="magma",
                   etiquetas=[f"pincel #{k}" for k in range(16)],
                   titulo="Capa 2 (14x14): ya no son bordes sueltos, son pedazos de trazo")
plt.show()

### 5.3 Las fotos que más encienden un pincel de la capa 3

Los pinceles de la tercera capa son difíciles de dibujar (cada uno mira 32 mapas
a la vez). Pero se los puede conocer **por lo que les gusta**: buscamos, entre
las 10.000 fotos de validación, las 8 que más encienden a cada uno, y marcamos
**dónde** se encendió. Es como conocer a alguien por las fotos que guarda.

El recuadro es aproximado: cada píxel de la capa 3 "ve" un cuadrado de unos
18×18 píxeles de la foto original.

In [ ]:
with torch.no_grad():
    _, _, a3_val = cnn.por_dentro(X_val)                 # (10000, 64, 7, 7)
maximo_por_foto = a3_val.flatten(2).max(dim=2)           # el máximo de cada mapa


def lo_que_le_gusta(pincel, n=8):
    valores = maximo_por_foto.values[:, pincel]
    mejores = valores.argsort(descending=True)[:n]
    fig, ejes = g.figura(13, 2.0, filas=1, columnas=n,
                         titulo=f"Las {n} fotos que más encienden al pincel #{pincel} de la capa 3")
    for ax, i in zip(ejes, mejores):
        ax.imshow(X_val[i, 0], cmap="gray")
        lugar = int(maximo_por_foto.indices[i, pincel])
        f, c = divmod(lugar, 7)
        ax.add_patch(plt.Rectangle((4 * c + 1.5 - 9, 4 * f + 1.5 - 9), 18, 18,
                                   fill=False, edgecolor=g.ACENTO, linewidth=2))
        ax.set_title(f"un {NOMBRES[int(y_val[i])]}")
        ax.set_xlim(-0.5, 27.5); ax.set_ylim(27.5, -0.5)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.show()


# los pinceles más "elegidos" por la capa que decide
importancia = cnn.decidir.weight.detach().abs().sum(dim=0)
for pincel in importancia.argsort(descending=True)[:4].tolist():
    lo_que_le_gusta(pincel)

> Vas a ver pinceles a los que les gustan **los rulos cerrados** (ceros, seises,
> ochos), otros que buscan **una raya horizontal arriba** (sietes, cincos),
> otros un **trazo vertical largo**. Ningún humano les dijo qué buscar. Y fijate
> que el recuadro aparece en distintos lugares de la foto: es el **mismo** pincel
> encontrando la **misma** pieza donde esté.

### 5.4 ¿Qué parte de la foto miró para decidir?

La pregunta que más importa para un investigador: cuando dijo "es un 7", ¿en qué
se fijó? Dos herramientas, las dos con cosas que ya sabés.

**Tapar y ver (oclusión).** Tapamos un cuadradito de la foto con negro y vemos
cuánto **baja** la probabilidad que el modelo le da a la respuesta correcta.
Movemos el cuadradito por toda la foto, como la ventana de la convolución. Donde
tapar duele mucho, ahí estaba mirando.

**Las pendientes, respecto de los píxeles (saliencia).** En la clase 3
calculamos la pendiente de la loss respecto de las perillas. Acá hacemos lo
mismo, pero respecto de **los píxeles de la foto**: ¿qué píxeles, si les
pusieras un poquito más de tinta, más subirían el puntaje de la respuesta? Es
`backward()`, el mismo de siempre, sobre la foto. Y como la pendiente sola
también se enciende en el fondo vacío, la multiplicamos por la tinta que hay en
cada píxel: queda **cuánto empujó cada trazo** hacia la respuesta.

In [ ]:
def mapa_de_oclusion(modelo, x, clase, lado=6, paso=2, relleno=0.0):
    """Cuánto baja la probabilidad de `clase` al tapar cada cuadradito de la foto x."""
    modelo.eval()
    _, alto, ancho = x.shape
    with torch.no_grad():
        p_original = torch.softmax(modelo(x.unsqueeze(0)), 1)[0, clase].item()
        tapadas, lugares = [], []
        for f in range(0, alto - lado + 1, paso):
            for c in range(0, ancho - lado + 1, paso):
                t = x.clone()
                t[:, f:f + lado, c:c + lado] = relleno
                tapadas.append(t)
                lugares.append((f, c))
        p = torch.softmax(modelo(torch.stack(tapadas)), 1)[:, clase]
    caida, veces = np.zeros((alto, ancho)), np.zeros((alto, ancho))
    for (f, c), pk in zip(lugares, p.tolist()):
        caida[f:f + lado, c:c + lado] += p_original - pk
        veces[f:f + lado, c:c + lado] += 1
    return caida / np.maximum(veces, 1)


def mapa_de_saliencia(modelo, x, clase):
    """Pendiente del puntaje de `clase` respecto de cada píxel, por la tinta del píxel."""
    modelo.eval()
    x = x.unsqueeze(0).clone().requires_grad_(True)   # "acordate de las cuentas"
    puntaje = modelo(x)[0, clase]
    puntaje.backward()                                # ANOTAR, como siempre
    empuje = x.grad * x                               # pendiente x tinta
    return empuje[0, 0].detach().numpy()


def mostrar_donde_miro(modelos, indices, relleno=0.0, nombres_clase=None):
    nombres_clase = nombres_clase or NOMBRES
    filas = 1 + 2 * len(modelos)
    fig, ejes = g.figura(1.9 * len(indices) + 1, 1.9 * filas + 0.6,
                         filas=filas, columnas=len(indices), squeeze=False,
                         titulo="Dónde miró cada modelo para decidir (más claro = más importante)")
    for j, i in enumerate(indices):
        x, clase = X_val[i], int(y_val[i])
        ejes[0, j].imshow(x[0], cmap="gray")
        ejes[0, j].set_title(f"un {nombres_clase[clase]}")
        for m, (nombre, modelo) in enumerate(modelos.items()):
            oc = mapa_de_oclusion(modelo, x, clase, relleno=relleno)
            sa = mapa_de_saliencia(modelo, x, clase)
            for fila, (mapa, que) in zip((1 + 2 * m, 2 + 2 * m),
                                         ((oc, "tapar"), (sa, "pendiente\nx tinta"))):
                ax = ejes[fila, j]
                ax.imshow(x[0], cmap="gray", alpha=0.35)
                ax.imshow(np.clip(mapa, 0, None), cmap="inferno", alpha=0.75)
                if j == 0:
                    ax.set_ylabel(f"{nombre}\n{que}", fontsize=9)
    for ax in ejes.flat:
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.show()


mostrar_donde_miro({"densa": densa, "CNN": cnn}, indices=[0, 1, 2, 3, 4, 5])

> **Cómo leerlo.** En las filas "tapar", lo claro es donde tapar la foto hizo
> **bajar** la confianza en la respuesta correcta: sin ese pedazo, el modelo
> duda. En las filas "pendiente x tinta", lo claro son los trazos que más
> empujaron hacia la respuesta. Compará densa y CNN sobre el mismo dígito:
> ¿miran el mismo pedazo? ¿Alguna se fija en algo que a vos no te parecería
> importante para reconocer ese número?
>
> **Y una advertencia que vale para toda tu carrera.** Estos mapas son
> **pistas, no la verdad**. Las dos herramientas a veces no coinciden, y las dos
> dependen de decisiones (el tamaño del cuadradito, con qué se tapa). Sirven
> para hacer preguntas y para detectar disparates, no para *probar* qué "piensa"
> un modelo. Lo retomamos en la clase 10.

### `TODO 4` — un error de la CNN, mirado por dentro

La celda de abajo agarra los primeros errores de la CNN y les hace el mapa de
"tapar" **para lo que dijo** (no para lo que era): ¿qué parte de la foto la
convenció de su respuesta equivocada?

Mirálos un minuto y escribí, para **uno** de ellos: qué dígito era, qué dijo la
CNN, y qué parte del trazo la llevó a equivocarse. ¿Te parece un error
"razonable" (un humano podría haberse equivocado igual) o un disparate?

In [ ]:
n_err = min(6, len(errores_cnn))
if n_err:
    fig, ejes = g.figura(2.1 * n_err + 1, 4.6, filas=2, columnas=n_err, squeeze=False,
                         titulo="Errores de la CNN: qué parte de la foto la convenció de lo que dijo")
    for j, i in enumerate(errores_cnn[:n_err].tolist()):
        x, era, dijo = X_val[i], int(y_val[i]), int(pred_cnn[i])
        ejes[0, j].imshow(x[0], cmap="gray")
        ejes[0, j].set_title(f"era {NOMBRES[era]}, dijo {NOMBRES[dijo]}", color=g.ERROR)
        ejes[1, j].imshow(x[0], cmap="gray", alpha=0.35)
        ejes[1, j].imshow(np.clip(mapa_de_oclusion(cnn, x, dijo), 0, None),
                          cmap="inferno", alpha=0.75)
        ejes[1, j].set_title(f"lo que lo llevó al {NOMBRES[dijo]}", fontsize=9)
    for ax in ejes.flat:
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.show()

In [ ]:
# TODO 4: elegí uno de los errores de arriba y escribí qué viste.
mi_lectura_del_error = "..."

print(mi_lectura_del_error)
_ = revisar(mi_lectura_del_error != "..." and len(mi_lectura_del_error) > 30,
        "listo: acabás de hacer análisis de errores mirando adentro del modelo",
        "escribí aunque sea una línea: qué era, qué dijo, y qué parte del trazo "
        "se iluminó en el mapa")

---

## 6. El sobre cerrado: el test, una sola vez

Comparamos modelos, elegimos, jugamos con variantes: todo eso con **validación**.
Ahora, y sólo ahora, abrimos el sobre. Es la regla de oro de la clase 4: el test
se toca **una vez**, al final, y el número que sale es el que se reporta, guste o
no.

In [ ]:
_, test_densa = evaluar(densa, cargador_test)
_, test_cnn = evaluar(cnn, cargador_test)
print(f"aciertos en test (10.000 fotos que nadie miró):")
print(f"   red densa  {test_densa:.1f} %")
print(f"   CNN        {test_cnn:.1f} %")
print(f"\nen validación habían dado {hist_densa['exactitud'][-1]:.1f} % y "
      f"{hist_cnn['exactitud'][-1]:.1f} %.")
print("Si el test da parecido a validación, la validación no te estaba mintiendo.")

---

## 7. Volvemos a ciencias sociales: un archivo fotográfico

Se terminó el sandbox. Ahora la situación real, la que aparece en una
investigación:

> Tenés un archivo fotográfico digitalizado — prensa gráfica, un fondo
> documental, fotos de campaña — y querés clasificar las imágenes en
> **"hay una movilización"** / **"no hay una movilización"**. Etiquetaste unas
> cuantas a mano y querés que el modelo haga el resto.

El corpus que vamos a usar es **sintético**: lo genera numpy con un proceso
generador que conocemos. Eso no es una limitación, es la gracia: **sabemos la
verdad**, y por eso podemos mostrar exactamente en qué se equivoca el modelo y
por qué. País ficticio, cero fotos de gente real.

Hay tres tipos de escena y dos etiquetas:

| escena | etiqueta | qué es |
|---|---|---|
| **multitud** | 1 | muchas cabecitas repartidas bajo el horizonte |
| **edificio** | 0 | un bloque grande con ventanas: una fachada, una tarima |
| **follaje** | 0 | racimos de manchitas: copas de árboles |

El **follaje está puesto a propósito**. Tiene la misma *textura* que una
multitud y ninguna persona adentro. Guardate esa carta: la vamos a cobrar en el
`TODO 5`, y
esta vez con los mapas de "dónde miró" en la mano.

In [ ]:
LADO = 32


def _suavizar(img):
    """Un desenfoque 3x3. Las fotos de archivo no tienen bordes de bisturí."""
    L = img.shape[0]
    k = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], float)
    k /= k.sum()
    p = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for j in range(3):
        for i in range(3):
            out += k[j, i] * p[j:j + L, i:i + L]
    return out


def archivo_visual(n, semilla, lado=LADO, prop_follaje=0.34):
    """Archivo fotográfico SINTÉTICO. Devuelve (imágenes, etiquetas, tipo)."""
    rng = np.random.default_rng(semilla)
    X = np.zeros((n, lado, lado))
    y = np.zeros(n, dtype=np.int64)
    tipo = []
    yy, xx = np.mgrid[0:lado, 0:lado]
    for k in range(n):
        base = rng.uniform(140, 205)
        img = np.tile(np.linspace(base, base - 40, lado), (lado, 1)).T
        r = rng.random()
        if k % 2 == 0:                                    # ---- multitud
            y[k] = 1
            tipo.append("multitud")
            hor = rng.integers(3, 18)
            img[hor:, :] = rng.uniform(110, 175)
            for _ in range(rng.integers(45, 120)):
                cy, cx = rng.integers(hor, lado - 1), rng.integers(0, lado)
                img[(yy - cy) ** 2 + (xx - cx) ** 2
                    <= rng.uniform(0.8, 2.0) ** 2] = rng.uniform(20, 90)
        elif r < prop_follaje:                            # ---- follaje
            y[k] = 0
            tipo.append("follaje")
            hor = rng.integers(3, 18)
            img[hor:, :] = rng.uniform(110, 175)
            for _ in range(rng.integers(2, 6)):
                by, bx = rng.integers(2, lado - 4), rng.integers(2, lado - 4)
                for _ in range(rng.integers(15, 40)):
                    cy = int(np.clip(by + rng.normal(0, 3.5), 0, lado - 1))
                    cx = int(np.clip(bx + rng.normal(0, 3.5), 0, lado - 1))
                    img[(yy - cy) ** 2 + (xx - cx) ** 2
                        <= rng.uniform(0.8, 2.0) ** 2] = rng.uniform(20, 90)
        else:                                             # ---- edificio
            y[k] = 0
            tipo.append("edificio")
            a, an = rng.integers(10, 20), rng.integers(12, 24)
            f0 = lado - a - rng.integers(0, 5)
            c0 = rng.integers(0, lado - an)
            img[f0:f0 + a, c0:c0 + an] = rng.uniform(60, 110)
            for fv in range(f0 + 2, f0 + a - 2, 4):
                for cv in range(c0 + 2, c0 + an - 2, 5):
                    img[fv:fv + 2, cv:cv + 3] = rng.uniform(170, 215)
        X[k] = np.clip(_suavizar(img) + rng.normal(0, 7, (lado, lado)), 0, 255)
    return X.astype(np.uint8), y, np.array(tipo)


X_arch, y_arch, tipo_arch = archivo_visual(400, 11)      # lo que etiquetamos
X_arch_val, y_arch_val, tipo_arch_val = archivo_visual(400, 22)         # lo que NUNCA vio

fig, _ = g.mostrar_imagenes(X_arch[:12], ncols=6,
                            etiquetas=[f"{t}\n(etiqueta {int(e)})"
                                       for t, e in zip(tipo_arch[:12], y_arch[:12])],
                            titulo="El archivo SINTÉTICO: 12 fotos de las 400 etiquetadas")
plt.show()

import collections
cuenta = collections.Counter(str(t) for t in tipo_arch)
print("qué hay en el archivo de entrenamiento:", dict(cuenta))

Entrenamos **la misma CNN** de antes. Fijate que no tocamos ni una línea de la
arquitectura, aunque ahora las imágenes son de 32×32 y no de 28×28: el
`AdaptiveMaxPool2d(1)` del final se acomoda solo. Eso también es un supuesto
hecho estructura.

Como son pocas fotos (400), las metemos en un `DataLoader` de lotes de 32 y
usamos el mismo `entrenar_una_epoca` de siempre.

In [ ]:
datos_arch = TensorDataset(a_tensor(X_arch), torch.tensor(y_arch, dtype=torch.long))
cargador_arch = DataLoader(datos_arch, batch_size=32, shuffle=True,
                           generator=torch.Generator().manual_seed(SEMILLA))
X_arch_val_t = a_tensor(X_arch_val)
y_arch_val_t = torch.tensor(y_arch_val, dtype=torch.long)

torch.manual_seed(SEMILLA)
modelo_archivo = RedConvolucional(2)
opt = torch.optim.Adam(modelo_archivo.parameters(), lr=2e-3)
for epoca in range(10):
    entrenar_una_epoca(modelo_archivo, cargador_arch, opt)

prediccion = predecir(modelo_archivo, X_arch_val_t).numpy()
print(f"exactitud sobre las 400 fotos que nunca vio: "
      f"{100 * (prediccion == y_arch_val).mean():.1f} %")

g.matriz_de_confusion(y_arch_val, prediccion,
                      etiquetas=["sin movilización", "hay movilización"],
                      nombre_positivo="hay movilización")
plt.show()

g.frase_de_metricas(y_arch_val, prediccion, fenomeno="fotos con movilización")

> **La matriz de confusión aparece acá por primera vez y no la vamos a explicar
> en detalle: la clase 9 vive de esto.** Por ahora quedate con la regla de la
> casa: prohibido reportar una métrica sin poder decirla en voz alta en
> castellano. Eso es lo que imprime `frase_de_metricas`.

### `TODO 5 (criterio)` — el ejercicio más importante del notebook

Una exactitud altísima (arriba del 95 %) no te dice **qué** aprendió el
modelo. Los errores sí.

La celda de abajo grafica los casos en los que se equivocó. Mirálos un minuto,
en silencio, antes de leer nada más, y escribí qué tienen en común.

*Pista: el conocimiento está en los errores, no en los aciertos. Y una segunda:
fijate si los errores se reparten parejo entre los tres tipos de escena o si se
amontonan en alguno. Mirá la exactitud **por tipo**, no sólo cuántos errores hay:
hay 200 multitudes y sólo unas 60 fotos de follaje.*

Esta vez tenés una herramienta más: debajo de cada error está su mapa de
"tapar", hecho para **lo que dijo el modelo**. Lo claro es lo que lo convenció.

In [ ]:
errores = np.where(prediccion != y_arch_val)[0]
print(f"{len(errores)} errores sobre {len(y_arch_val)} fotos\n")

n_mostrar = min(6, len(errores))
if n_mostrar:
    corto = {0: "NO", 1: "SÍ"}          # SÍ = hay movilización
    fig, ejes = g.figura(2.2 * n_mostrar + 1, 4.8, filas=2, columnas=n_mostrar, squeeze=False,
                         titulo="Los errores, y qué los convenció  ·  SÍ = hay movilización")
    for j, k in enumerate(errores[:n_mostrar]):
        x = X_arch_val_t[k]
        dijo = int(prediccion[k])
        ejes[0, j].imshow(x[0], cmap="gray")
        ejes[0, j].set_title(f"{tipo_arch_val[k]}: dijo {corto[dijo]}", color=g.ERROR, fontsize=9)
        mapa = mapa_de_oclusion(modelo_archivo, x, dijo, lado=8, relleno=float(x.mean()))
        ejes[1, j].imshow(x[0], cmap="gray", alpha=0.35)
        ejes[1, j].imshow(np.clip(mapa, 0, None), cmap="inferno", alpha=0.75)
        ejes[1, j].set_title("lo que lo convenció", fontsize=9)
    for ax in ejes.flat:
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.show()

# la tabla que hay que mirar: ¿los errores se reparten parejo?
print(f"{'escena':10s} {'cuántas':>8s} {'exactitud':>10s} {'errores':>8s}")
exact_tipo = {}
for t in ["multitud", "follaje", "edificio"]:
    sel = tipo_arch_val == t
    exact_tipo[t] = 100 * (prediccion[sel] == y_arch_val[sel]).mean()
    print(f"{t:10s} {sel.sum():8d} {exact_tipo[t]:9.1f}% "
          f"{int((prediccion[sel] != y_arch_val[sel]).sum()):8d}")

peor = min(exact_tipo, key=exact_tipo.get)
if peor == "multitud":
    print("\nOJO: en esta corrida el que peor sale es «multitud», no el follaje. Es la")
    print("misma confusión mirada del otro lado: el modelo se pierde las multitudes")
    print("con pocas cabecitas. Los errores siguen estando en las escenas con")
    print("manchitas, casi nunca en los edificios.")
elif peor != "follaje":
    print(f"\nOJO: en esta corrida el que peor sale es «{peor}». No es lo que pasa en")
    print("la corrida de las slides: mirá las figuras de los errores y compará.")

In [ ]:
# TODO 5: ¿qué tienen en común los errores? ¿Qué está midiendo REALMENTE
# el modelo cuando dice "hay movilización"?
que_mide_realmente = "..."

print(que_mide_realmente)
_ = revisar(que_mide_realmente != "..." and len(que_mide_realmente) > 30,
        "escrito. Ese párrafo es la sección de limitaciones de tu paper",
        "escribilo igual, aunque sea una hipótesis. Pista: mirá la columna "
        "'exactitud' por tipo de escena y preguntate qué tienen en común una "
        "multitud y una copa de árbol vistas desde lejos y en baja resolución")

> **Lo que acaba de pasar.** El modelo acierta muchísimo *en promedio*, y sin
> embargo los errores **no se reparten al azar**: se amontonan en las escenas con
> manchitas —sobre todo en el follaje, que es el que peor sale— y muy poco
> en los edificios.
> Traducido: el modelo no aprendió "hay gente". Aprendió *"hay muchas manchitas
> chiquitas repartidas"*, que en este archivo casi siempre coincide con gente.
>
> Eso tiene un nombre en ciencias sociales y ya lo conocés: es un problema de
> **validez de constructo**. El indicador correlaciona con lo que te interesa,
> pero no *es* lo que te interesa, y el día que la correlación se rompe —otro
> archivo, otra ciudad, otra época, más árboles— el modelo se rompe sin avisar.
>
> Fijate además que esto es la contracara exacta de la ventaja que celebramos en
> la sección 4: la CNN es invariante a la posición, así que le da igual que las
> manchitas estén arriba (copas) o abajo (cabezas). **El supuesto que te salva
> es el mismo que te hunde.**
>
> Y fijate lo que mostraron los mapas de "tapar": en las fotos de follaje que
> confundió, lo que lo convenció de decir "SÍ" fueron **las copas de los
> árboles**. La herramienta para mirar adentro no te dice qué tiene que medir
> tu modelo. Te dice, a veces, que está midiendo otra cosa.

---

## 8. Qué te llevás

1. **La arquitectura es una teoría sobre el mundo.** La CNN afirma que lo que
   importa es local y que un patrón vale igual en cualquier posición. Lo
   mediste: con la foto corrida 4 píxeles la densa se desplomó y la CNN aguantó;
   y con el `TODO 3` encontraste **qué pieza** de la CNN pone ese supuesto.
2. **Un filtro son nueve números y un mapa de activación es un dibujo.** Lo
   escribiste con dos `for` y después comprobaste que `nn.Conv2d` hace la misma
   cuenta. No hay magia adentro; lo único nuevo es la *forma* de la máquina. El
   loop de entrenamiento fue **el mismo** de la clase 4 para las dos redes.
3. **Se puede mirar adentro, con humildad.** Los pinceles, los mapas, las fotos
   que encienden cada pincel y los mapas de "dónde miró" son pistas. Sirven para
   encontrar disparates y para escribir la sección de limitaciones, no para
   probar qué "entiende" un modelo.
4. **La exactitud global no te dice qué está midiendo el modelo. Los errores
   sí.** Y cuando los errores se concentran en un subgrupo, ahí está tu sección
   de limitaciones.

**Las piezas de PyTorch que aparecieron hoy** (vas a verlas en cualquier
tutorial): `TensorDataset`, `random_split`, `DataLoader`, una clase que hereda
de `nn.Module` con `__init__` y `forward`, `nn.Conv2d`, `nn.MaxPool2d`,
`nn.AdaptiveMaxPool2d`, `nn.CrossEntropyLoss`, `torch.softmax`, y
`requires_grad_` + `backward()` sobre una foto para la saliencia.

**En tu propia investigación, desde ya.**

Si en tu investigación hay imágenes —archivo fotográfico, prensa gráfica,
material etnográfico visual, actas escaneadas— podés empezar desde ya con
tres cosas y **ninguna** requiere entrenar nada (no es tarea para la clase
que viene: es cómo se usa esto en tu trabajo):

1. Etiquetá a mano 100 imágenes de cada categoría. Es una tarde, y sin eso no
   hay proyecto. Con esas pocas etiquetas y un modelo que ya entrenó otro
   (transferencia) se puede llegar lejos: en la clase 9 vemos esa receta con
   texto, y con imágenes es la misma.
2. Definí **antes** la partición: si tus fotos vienen de distintos archivos,
   ciudades o épocas, partí por ahí y no al azar (clase 4).
3. Escribí en una línea cuál es tu **baseline**: ¿qué acierta el "siempre digo
   que no"? Si es 85 %, un modelo del 88 % no es una noticia.

### Las visualizaciones de esta clase

Están en `clases/05-imagenes-y-convoluciones/viz/` y abren con doble clic, sin
internet. Si algo de hoy quedó flojo, la que conviene volver a abrir es:

- `vecindad-de-pixeles.html` — el barajado de píxeles (un experimento que no
  hicimos en el notebook) y la foto corrida.
- `kernel-deslizante.html` — los nueve números, editables, sobre ocho imágenes
  (cuatro prendas y cuatro esquemas de archivo).
- `jerarquia.html` — de qué está hecha una «rueda».
- `supuestos-de-arquitectura.html` — **la que conviene tener a mano el día que
  elijas un modelo para tus propios datos.**

Y si vas a **leer** un paper que clasifica imágenes, las tres preguntas son:
¿qué está midiendo realmente?, ¿valen acá los supuestos de la arquitectura?,
¿qué pasa fuera de la región donde se entrenó?

### El puente a la clase 6

La estructura del espacio pide convoluciones. Pero el texto no es una grilla:
es una **secuencia**, y el significado de una palabra depende de palabras que
pueden estar muy lejos. *"La comisión rechazó al ministro"* y *"el ministro
rechazó a la comisión"* usan las mismas palabras con contenido y dicen lo
contrario.

La clase que viene: cómo se convierte un discurso en números **sin perder el
orden**.

**Para la clase que viene:** terminar la sección 7 si no llegaste, y las
preguntas 1-7 y 10 del cuestionario (menos de una hora en total). La lectura de
Jean et al. (2016) está en `referencias.md` y es recomendada, no obligatoria.